## Dividend Investing Strategy

In [20]:
# import libraries
import pandas as pd
import numpy as np
import yfinance as yf
import math
from scipy import stats

## Load the Stock List

In [21]:
tickers = pd.read_csv('malaysia_stocks.csv')
tickers.head()

,Company Name,Ticker
0,Hong Leong Bank Berhad,5819.KL
1,Malayan Banking Berhad,1155.KL
2,Hong Leong Financial Group Berhad,1082.KL
3,QL Resources Berhad,7084.KL
4,Maxis Berhad,6012.KL


## Weighted Scoring Model to Find the Final Score

In [22]:
def create_dividend_df(tickers):
    columns = [
        "Ticker", 
        "Dividend Yield(%)", 
        "Dividend Rate",
        "Payout Ratio(%)", 
        "5 Year Avg Dividend Yield(%)", 
        "Earning Growth(%)"
        
    ]
    
    dividend_df = pd.DataFrame(columns=columns)
    
    for stock in tickers:
        ticker = yf.Ticker(stock)
        info = ticker.info 
        
        dividend_yield = info.get("dividendYield", np.nan) * 100 if info.get("dividendYield") else np.nan
        dividend_rate = info.get("dividendRate", np.nan) 
        payout_ratio = info.get("payoutRatio", np.nan) * 100 if info.get("payoutRatio") else np.nan
        five_year_avg_dividend_yield = info.get("fiveYearAvgDividendYield", np.nan) * 100 if info.get("fiveYearAvgDividendYield") else np.nan
        earning_growth = info.get("earningsGrowth", np.nan) * 100 if info.get("earningsGrowth") else np.nan
        
        dividend_df.loc[len(dividend_df)] = [stock, dividend_yield, dividend_rate, payout_ratio, five_year_avg_dividend_yield, earning_growth]
        
    ## we have got our df with dividend information
    
    numeric_cols = [
          "Dividend Yield(%)", 
        "Dividend Rate",
        "Payout Ratio(%)", 
        "5 Year Avg Dividend Yield(%)", 
        "Earning Growth(%)"
    ]
    

    for col in numeric_cols:
        if col == "Payout Ratio":
            dividend_df[col+ " Normalised"] = 1 - ((dividend_df[col] - dividend_df[col].min()) / (dividend_df[col].max() - dividend_df[col].min()))
        else:
            dividend_df[col+ " Normalised"] = (dividend_df[col] - dividend_df[col].min()) / (dividend_df[col].max() - dividend_df[col].min())
        
    return dividend_df

## Evaluate and Compare Stocks Based on Dividend

In [23]:
tickers_list = tickers["Ticker"].values.tolist()
dividend_df = create_dividend_df(tickers_list)
dividend_df

,Ticker,Dividend Yield(%),Dividend Rate,Payout Ratio(%),5 Year Avg Dividend Yield(%),Earning Growth(%),Dividend Yield(%) Normalised,Dividend Rate Normalised,Payout Ratio(%) Normalised,5 Year Avg Dividend Yield(%) Normalised,Earning Growth(%) Normalised
0,5819.KL,465.0,0.96,34.250000,280.0,4.7,0.669829,0.330909,0.084408,0.334454,0.064745
1,1155.KL,603.0,0.60,71.589994,676.0,3.8,0.931689,0.200000,0.298378,1.000000,0.063626
2,1082.KL,423.0,0.72,19.520001,255.0,5.9,0.590133,0.243636,0.000000,0.292437,0.066236
3,7084.KL,112.0,0.05,39.330003,81.0,-6.4,0.000000,0.000000,0.113518,0.000000,0.050951
4,6012.KL,437.0,0.16,91.890000,431.0,11.8,0.616698,0.040000,0.414704,0.588235,0.073568
5,1961.KL,265.0,0.11,40.800000,273.0,25.9,0.290323,0.021818,0.121941,0.322689,0.091090
6,2445.KL,293.0,0.60,60.790000,336.0,42.1,0.343454,0.200000,0.236491,0.428571,0.111222
7,6033.KL,338.0,0.64,77.870005,421.0,-4.0,0.428843,0.214545,0.334365,0.571429,0.053933
8,5347.KL,378.0,0.50,62.450004,415.0,-20.0,0.504744,0.163636,0.246003,0.561345,0.034050
9,6947.KL,415.0,0.15,120.430000,356.0,8.1,0.574953,0.036364,0.578248,0.462185,0.068970


## Rank Stocks Based on Dividend Performance

In [24]:
weights = {
        "Dividend Yield(%) Normalised": 0.3,
        "Dividend Rate Normalised": 0.2,
        "Payout Ratio(%) Normalised": 0.2,
        "5 Year Avg Dividend Yield(%) Normalised": 0.2,
        "Earning Growth(%) Normalised": 0.1
    }

dividend_df["Dividend Score"] = dividend_df[[col for col in weights.keys()]].mul(list(weights.values())).sum(axis = 1)
dividend_df

,Ticker,Dividend Yield(%),Dividend Rate,Payout Ratio(%),5 Year Avg Dividend Yield(%),Earning Growth(%),Dividend Yield(%) Normalised,Dividend Rate Normalised,Payout Ratio(%) Normalised,5 Year Avg Dividend Yield(%) Normalised,Earning Growth(%) Normalised,Dividend Score
0,5819.KL,465.0,0.96,34.250000,280.0,4.7,0.669829,0.330909,0.084408,0.334454,0.064745,0.357377
1,1155.KL,603.0,0.60,71.589994,676.0,3.8,0.931689,0.200000,0.298378,1.000000,0.063626,0.585545
2,1082.KL,423.0,0.72,19.520001,255.0,5.9,0.590133,0.243636,0.000000,0.292437,0.066236,0.290878
3,7084.KL,112.0,0.05,39.330003,81.0,-6.4,0.000000,0.000000,0.113518,0.000000,0.050951,0.027799
4,6012.KL,437.0,0.16,91.890000,431.0,11.8,0.616698,0.040000,0.414704,0.588235,0.073568,0.400954
5,1961.KL,265.0,0.11,40.800000,273.0,25.9,0.290323,0.021818,0.121941,0.322689,0.091090,0.189495
6,2445.KL,293.0,0.60,60.790000,336.0,42.1,0.343454,0.200000,0.236491,0.428571,0.111222,0.287171
7,6033.KL,338.0,0.64,77.870005,421.0,-4.0,0.428843,0.214545,0.334365,0.571429,0.053933,0.358114
8,5347.KL,378.0,0.50,62.450004,415.0,-20.0,0.504744,0.163636,0.246003,0.561345,0.034050,0.349025
9,6947.KL,415.0,0.15,120.430000,356.0,8.1,0.574953,0.036364,0.578248,0.462185,0.068970,0.394742


## Sort According to Dividend Score

In [25]:
dividend_df = dividend_df.sort_values(by = "Dividend Score", ascending = False)
dividend_df.head(10)

,Ticker,Dividend Yield(%),Dividend Rate,Payout Ratio(%),5 Year Avg Dividend Yield(%),Earning Growth(%),Dividend Yield(%) Normalised,Dividend Rate Normalised,Payout Ratio(%) Normalised,5 Year Avg Dividend Yield(%) Normalised,Earning Growth(%) Normalised,Dividend Score
1,1155.KL,603.0,0.60,71.589994,676.0,3.8,0.931689,0.200000,0.298378,1.000000,0.063626,0.585545
15,1066.KL,639.0,0.43,58.380000,607.0,10.3,1.000000,0.138182,0.222681,0.884034,0.071704,0.556150
11,4197.KL,602.0,0.13,43.190000,508.0,757.3,0.929791,0.029091,0.135637,0.717647,1.000000,0.555412
23,3816.KL,486.0,0.36,151.260000,468.0,-14.1,0.709677,0.112727,0.754914,0.650420,0.041382,0.520654
19,4707.KL,287.0,2.80,105.100000,227.0,19.8,0.332068,1.000000,0.490402,0.245378,0.083509,0.455127
24,1023.KL,527.0,0.40,55.979997,436.0,-4.0,0.787476,0.127273,0.208928,0.596639,0.053933,0.428204
4,6012.KL,437.0,0.16,91.890000,431.0,11.8,0.616698,0.040000,0.414704,0.588235,0.073568,0.400954
9,6947.KL,415.0,0.15,120.430000,356.0,8.1,0.574953,0.036364,0.578248,0.462185,0.068970,0.394742
17,1295.KL,500.0,0.22,56.320000,402.0,-0.7,0.736243,0.061818,0.210876,0.539496,0.058034,0.389114
20,5681.KL,387.0,0.88,77.390000,332.0,-3.9,0.521822,0.301818,0.331614,0.421849,0.054057,0.373008
